In [ ]:
Berikut versi **copas langsung ke Kaggle Notebook**.

```markdown
# Mini MaleVis Generator

Proyek sederhana untuk mengubah file binary menjadi citra RGB.

Alur:

File Binary → Byte → RGB Pixel → RGB Image → Resize → Save Image

Program ini hanya membaca file sebagai binary.  
Program tidak menjalankan file, tidak melakukan reverse engineering, tidak melakukan analisis malware, dan tidak melakukan klasifikasi.
```

```python
# ============================================================
# CELL 1 — Import Library
# ============================================================

import os
import math
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
```

```python
# ============================================================
# CELL 2 — Membuat Struktur Folder
# ============================================================

BASE_DIR = Path("/kaggle/working/mini-malevis-generator")

SAMPLE_DIR = BASE_DIR / "sample"
ORIGINAL_SAMPLE_DIR = SAMPLE_DIR / "original"
MALWARE_SAMPLE_DIR = SAMPLE_DIR / "malware"

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_ORIGINAL_DIR = OUTPUT_DIR / "original"
OUTPUT_224_DIR = OUTPUT_DIR / "resized_224"
OUTPUT_300_DIR = OUTPUT_DIR / "resized_300"
OUTPUT_VIS_DIR = OUTPUT_DIR / "visualization"

folders = [
    ORIGINAL_SAMPLE_DIR,
    MALWARE_SAMPLE_DIR,
    OUTPUT_ORIGINAL_DIR,
    OUTPUT_224_DIR,
    OUTPUT_300_DIR,
    OUTPUT_VIS_DIR,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Struktur folder berhasil dibuat:")
print(BASE_DIR)
```

````markdown
## Cara Memasukkan File

Upload file legal atau file binary ke Kaggle.

Contoh file:

- `.exe`
- `.dll`
- `.bin`
- `.zip`
- file binary lain

Letakkan file di folder:

```text
/kaggle/working/mini-malevis-generator/sample/original/
````

atau jika file penelitian malware sudah tersedia:

```text
/kaggle/working/mini-malevis-generator/sample/malware/
```

Program hanya membaca file sebagai binary.

````

```python
# ============================================================
# CELL 3 — Cek File Sample
# ============================================================

def list_files(folder):
    files = list(folder.glob("*"))
    if len(files) == 0:
        print(f"Tidak ada file di: {folder}")
    else:
        print(f"File ditemukan di: {folder}")
        for i, file in enumerate(files, start=1):
            print(f"{i}. {file.name}")

print("=== Folder Original ===")
list_files(ORIGINAL_SAMPLE_DIR)

print("\n=== Folder Malware Research Sample ===")
list_files(MALWARE_SAMPLE_DIR)
````

```python
# ============================================================
# CELL 4 — Pilih File Input
# ============================================================

# GANTI PATH INI SESUAI FILE ANDA
# Contoh:
# input_file = ORIGINAL_SAMPLE_DIR / "putty.exe"
# input_file = MALWARE_SAMPLE_DIR / "sample_01.bin"

input_file = ORIGINAL_SAMPLE_DIR / "nama_file_anda.exe"

if not input_file.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan: {input_file}\n"
        "Silakan upload file terlebih dahulu dan ubah nama file pada variabel input_file."
    )

print("File input ditemukan:")
print(input_file)
```

```python
# ============================================================
# CELL 5 — Fungsi Membaca File Binary
# ============================================================

def read_binary_file(file_path):
    with open(file_path, "rb") as f:
        data = f.read()
    return data


binary_data = read_binary_file(input_file)

file_name = input_file.stem
file_size = input_file.stat().st_size
total_bytes = len(binary_data)

print("Nama File   :", input_file.name)
print("Ukuran File :", file_size, "bytes")
print("Total Bytes :", total_bytes)
```

```python
# ============================================================
# CELL 6 — Konversi Byte Menjadi RGB Pixel
# ============================================================

def binary_to_rgb_array(binary_data):
    byte_array = np.frombuffer(binary_data, dtype=np.uint8)

    total_bytes = len(byte_array)

    remainder = total_bytes % 3
    if remainder != 0:
        padding_size = 3 - remainder
        byte_array = np.pad(byte_array, (0, padding_size), mode="constant", constant_values=0)
    else:
        padding_size = 0

    rgb_pixels = len(byte_array) // 3

    width = math.ceil(math.sqrt(rgb_pixels))
    height = math.ceil(rgb_pixels / width)

    total_pixels_needed = width * height
    current_pixels = rgb_pixels

    if current_pixels < total_pixels_needed:
        extra_pixels = total_pixels_needed - current_pixels
        extra_bytes = extra_pixels * 3
        byte_array = np.pad(byte_array, (0, extra_bytes), mode="constant", constant_values=0)

    image_array = byte_array.reshape((height, width, 3))

    return image_array, rgb_pixels, width, height, padding_size


rgb_array, rgb_pixels, width, height, padding_size = binary_to_rgb_array(binary_data)

print("RGB Pixels          :", rgb_pixels)
print("Original Resolution :", width, "x", height)
print("Padding Byte        :", padding_size)
print("Array Shape         :", rgb_array.shape)
```

```python
# ============================================================
# CELL 7 — Membuat dan Menyimpan Gambar Original
# ============================================================

original_image = Image.fromarray(rgb_array, mode="RGB")

original_output_path = OUTPUT_ORIGINAL_DIR / f"{file_name}_original.png"
original_image.save(original_output_path)

print("Original image saved:")
print(original_output_path)
```

```python
# ============================================================
# CELL 8 — Resize ke 224x224 dan 300x300
# ============================================================

image_224 = original_image.resize((224, 224))
image_300 = original_image.resize((300, 300))

output_224_path = OUTPUT_224_DIR / f"{file_name}_224x224.png"
output_300_path = OUTPUT_300_DIR / f"{file_name}_300x300.png"

image_224.save(output_224_path)
image_300.save(output_300_path)

print("224x224 image saved:")
print(output_224_path)

print("\n300x300 image saved:")
print(output_300_path)
```

```python
# ============================================================
# CELL 9 — Visualisasi Original, 224x224, dan 300x300
# ============================================================

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(original_image)
plt.title(f"Original\n{width} x {height}")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(image_224)
plt.title("Resized\n224 x 224")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(image_300)
plt.title("Resized\n300 x 300")
plt.axis("off")

plt.tight_layout()

visualization_path = OUTPUT_VIS_DIR / f"{file_name}_comparison.png"
plt.savefig(visualization_path, dpi=150)
plt.show()

print("Visualization saved:")
print(visualization_path)
```

```python
# ============================================================
# CELL 10 — Summary Output
# ============================================================

def format_size(size_bytes):
    if size_bytes < 1024:
        return f"{size_bytes} Bytes"
    elif size_bytes < 1024 ** 2:
        return f"{size_bytes / 1024:.2f} KB"
    elif size_bytes < 1024 ** 3:
        return f"{size_bytes / (1024 ** 2):.2f} MB"
    else:
        return f"{size_bytes / (1024 ** 3):.2f} GB"


print("====================================")
print("Mini MaleVis Generator")
print("====================================")
print("Input File          :", input_file)
print("File Size           :", format_size(file_size))
print("Total Bytes         :", total_bytes)
print("RGB Pixels          :", rgb_pixels)
print("Original Resolution :", f"{width} x {height}")
print("224 Resolution      : 224 x 224")
print("300 Resolution      : 300 x 300")
print("Output Location     :", OUTPUT_DIR)
print("Completed")
print("====================================")
```

```markdown
## Catatan Keamanan

Program ini hanya melakukan:

- membaca file sebagai binary
- mengubah byte menjadi RGB
- membuat gambar
- resize gambar
- menyimpan gambar
- menampilkan visualisasi

Program ini tidak melakukan:

- menjalankan executable
- reverse engineering
- static analysis
- dynamic analysis
- deteksi malware
- klasifikasi malware
```

Setelah semua cell dibuat, jalankan dari atas ke bawah. Bagian yang perlu Anda ubah hanya ini:

```python
input_file = ORIGINAL_SAMPLE_DIR / "nama_file_anda.exe"
```
